# Normalize Data

In [2]:
import pandas as pd
import numpy as np
!pip install xlrd
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]


Uplode user table

In [11]:
users = pd.read_excel('/data/notebook_files/social_media_campaign_analysis_sorce.xlsx', sheet_name='users')
users

,user_id,user_gender,user_age,country,location,interests
0,a2474,Female,24,United Kingdom,New Mariomouth,"fitness, health"
1,14100000,Male,21,Germany,Danielsfort,"food, fitness, lifestyle"
2,34db0,Male,27,Australia,Vincentchester,"fashion, news"
3,20d08,Female,28,India,Lisaport,"health, news, finance"
4,9e830,Male,28,United States,Brownmouth,"health, photography, lifestyle"
...,...,...,...,...,...,...
9995,24364,Male,18,United States,Curtisside,"travel, fashion, art"
9996,68b82,Male,24,Mexico,Brownland,finance
9997,39f39,Male,29,United States,Watersburgh,health
9998,0b8e7,Male,31,United Kingdom,South Kenneth,"art, fashion"


Uplode Ads table

In [12]:
ads = pd.read_excel('/data/notebook_files/social_media_campaign_analysis_sorce.xlsx', sheet_name='ads')
ads

,ad_id,campaign_id,ad_platform,ad_type,target_gender,target_age_group,target_interests
0,1,28,Facebook,Video,Female,35-44,"art, technology"
1,2,33,Facebook,Stories,All,25-34,"travel, photography"
2,3,20,Instagram,Carousel,All,25-34,technology
3,4,28,Facebook,Stories,Female,25-34,news
4,5,24,Instagram,Image,Female,25-34,news
...,...,...,...,...,...,...,...
195,196,12,Facebook,Stories,Male,35-44,"gaming, sports"
196,197,9,Facebook,Stories,All,All,"lifestyle, gaming"
197,198,34,Facebook,Video,All,35-44,"fitness, lifestyle"
198,199,15,Instagram,Video,Male,25-34,"art, news"


In [14]:
def disttinct_intrests(interests):
    res = ", ".join(interests)
    return set(res.split(", "))

In [15]:
users_interests = disttinct_intrests(users['interests'])
ads_interests = disttinct_intrests(ads['target_interests'])
all_interests = users_interests | ads_interests

interests = pd.DataFrame({'interest_id': range(1, len(all_interests) + 1), 'interest': list(all_interests)})
interests

,interest_id,interest
0,1,travel
1,2,news
2,3,lifestyle
3,4,sports
4,5,fashion
5,6,fitness
6,7,food
7,8,health
8,9,photography
9,10,finance


In [16]:
def create_interests_fact_table(dim_df, interest_df, dim_id, dim_interest, interest_key):
    fact_df = dim_df.assign(**{dim_interest: dim_df[dim_interest].str.split(',')}).explode(dim_interest)
    fact_df[dim_interest] = fact_df[dim_interest].str.strip()
    fact_df = fact_df[[dim_id,dim_interest]].merge(interest_df, left_on=dim_interest, right_on=interest_key)
    fact_df = fact_df.drop([dim_interest, interest_key], axis=1)
    return fact_df

In [17]:
user_interest_test = create_interests_fact_table(ads, interests, 'ad_id','target_interests', 'interest')
user_interest_test

,ad_id,interest_id
0,1,11
1,1,12
2,2,1
3,2,9
4,3,12
...,...,...
295,198,6
296,198,3
297,199,11
298,199,2


In [18]:
user_interest = (users.assign(interest=users['interests'].str.split(',')).explode('interest'))
user_interest['interest'] = user_interest['interest'].str.strip()
user_interest = user_interest[['user_id', 'interest']]
user_interest = user_interest.merge(interests, on='interest')
user_interest.drop('interest',axis=1,  inplace=True)
user_interest

,user_id,interest_id
0,a2474,6
1,a2474,8
2,14100000,7
3,14100000,6
4,14100000,3
...,...,...
19929,68b82,10
19930,39f39,8
19931,0b8e7,11
19932,0b8e7,5


In [19]:
ad_interest = (ads.assign(target_interests=ads['target_interests'].str.split(',')).explode('target_interests'))

ad_interest['target_interests'] = ad_interest['target_interests'].str.strip()

ad_interest = ad_interest[['ad_id', 'target_interests']]
ad_interest = ad_interest.merge(interests, left_on='target_interests', right_on='interest')
ad_interest.drop(['target_interests','interest'],axis=1,  inplace=True)
ad_interest

,ad_id,interest_id
0,1,11
1,1,12
2,2,1
3,2,9
4,3,12
...,...,...
295,198,6
296,198,3
297,199,11
298,199,2


In [20]:
ad_interest = (ads.assign(target_interests=ads['target_interests'].str.split(',')).explode('target_interests'))

ad_interest['target_interests'] = ad_interest['target_interests'].str.strip()

ad_interest = ad_interest[['ad_id', 'target_interests']]
ad_interest

,ad_id,target_interests
0,1,art
0,1,technology
1,2,travel
1,2,photography
2,3,technology
...,...,...
197,198,fitness
197,198,lifestyle
198,199,art
198,199,news


In [21]:
ad_interest = ad_interest.merge(interests, left_on='target_interests', right_on='interest')
ad_interest.drop(['target_interests','interest'],axis=1,  inplace=True)
ad_interest

,ad_id,interest_id
0,1,11
1,1,12
2,2,1
3,2,9
4,3,12
...,...,...
295,198,6
296,198,3
297,199,11
298,199,2


In [22]:
pairs = [
    # fashion
    ("fashion", "lifestyle"),
    ("fashion", "art"),

    # news
    ("news", "finance"),
    ("news", "technology"),
    ("news", "sports"),

    # health
    ("health", "sports"),
    ("health", "fitness"),
    ("health", "food"),
    ("health", "lifestyle"),

    # lifestyle
    ("lifestyle", "health"),
    ("lifestyle", "fitness"),
    ("lifestyle", "fashion"),
    ("lifestyle", "travel"),

    # fitness
    ("fitness", "health"),
    ("fitness", "sports"),
    ("fitness", "food"),
    ("fitness", "lifestyle"),

    # technology
    ("technology", "gaming"),
    ("technology", "finance"),
    ("technology", "news"),
    ("technology", "photography"),

    # art
    ("art", "fashion"),
    ("art", "photography"),

    # travel
    ("travel", "photography"),
    ("travel", "lifestyle"),

    # food
    ("food", "lifestyle"),
    ("food", "health"),
    ("food", "fitness"),

    # sports
    ("sports", "fitness"),
    ("sports", "health"),
    ("sports", "news"),
    ("sports", "gaming"),

    # finance
    ("finance", "technology"),
    ("finance", "news"),

    # photography
    ("photography", "travel"),
    ("photography", "art"),
    ("photography", "technology"),

    # gaming
    ("gaming", "technology"),
    ("gaming", "sports"),

]
sub_interests = pd.DataFrame(pairs, columns=["interest", "related_interest"])

sub_interests = (sub_interests.merge(interests, on="interest")
    .rename(columns={"interest_id": "interest_id"}).merge(interests,
        left_on="related_interest", right_on="interest", suffixes=("", "_related"))
    [["interest_id", "interest_id_related"]]
)
sub_interests.sort_values("interest_id", inplace=True)
sub_interests

,interest_id,interest_id_related
24,1,3
23,1,9
2,2,10
3,2,12
4,2,4
9,3,8
10,3,6
11,3,5
12,3,1
30,4,2


In [23]:
users.drop('interests', axis=1, inplace=True)
ads.drop('target_interests', axis=1, inplace=True)
campaigns = pd.read_excel('/data/notebook_files/social_media_campaign_analysis_sorce.xlsx', sheet_name='campaigns')
ad_events = pd.read_excel('/data/notebook_files/social_media_campaign_analysis_sorce.xlsx', sheet_name='ad_events')

In [17]:

tables = [users, campaigns, ads, interests, ad_events, user_interest, ad_interest, sub_interests]
sheet_names = ['users', 'campaigns', 'ads', 'interests', 'ad_events', 'user_interest',
               'ad_interest', 'sub_interests']
path = 'Social_Media_Campaign_Analysis.xlsx'
er =pd.ExcelWriter(path)
for i in range(len(tables)):
    tables[i].to_excel(er, sheet_name=sheet_names[i], index = False)
er.close()

# Data Cleaning

Uplode Data

In [22]:
users = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis.xlsx', sheet_name='users')
ads = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis.xlsx', sheet_name='ads')
campaigns = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis.xlsx', sheet_name='campaigns')
ad_events = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis.xlsx', sheet_name='ad_events')
interests = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis.xlsx', sheet_name='interests')
user_interest = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis.xlsx', sheet_name='user_interest')
ad_interest = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis.xlsx', sheet_name='ad_interest')
sub_interests = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis.xlsx', sheet_name='sub_interests')

### step 1
Remove duplicate entries for the same user ID from the users table.

In [23]:
duplicates = users['user_id'].duplicated(keep=False)
duplicate_user_ids = users.loc[duplicates,'user_id'].unique()
users_clean = users.loc[~users['user_id'].isin(duplicate_user_ids)]
users_clean

,user_id,user_gender,user_age,country,location
0,a2474,Female,24,United Kingdom,New Mariomouth
1,14100000,Male,21,Germany,Danielsfort
2,34db0,Male,27,Australia,Vincentchester
3,20d08,Female,28,India,Lisaport
4,9e830,Male,28,United States,Brownmouth
...,...,...,...,...,...
9995,24364,Male,18,United States,Curtisside
9996,68b82,Male,24,Mexico,Brownland
9997,39f39,Male,29,United States,Watersburgh
9998,0b8e7,Male,31,United Kingdom,South Kenneth


In [24]:
removed = users.shape[0] -  users_clean.shape[0]
print(f'{removed} records were removed, {removed*100/users.shape[0]}% from the data')

107 records were removed, 1.07% from the data


### step 2
Remove duplicate entries for the same user ID from the ad_events table.

In [25]:
ad_events_clean = ad_events.loc[~ad_events['user_id'].isin(duplicate_user_ids)]
ad_events_clean

,event_id,ad_id,user_id,timestamp,event_type
0,1,197,2359b,2025-07-26 00:19:56,Like
1,2,51,f9c67,2025-06-15 08:28:07,Share
2,3,46,5b868,2025-06-27 00:40:02,Impression
3,4,166,3d440,2025-06-05 19:20:45,Impression
4,5,52,68f1a,2025-07-22 08:30:29,Impression
...,...,...,...,...,...
399995,399996,132,3cb8c,2025-08-01 22:36:54,Impression
399996,399997,200,fe0e3,2025-05-31 14:53:18,Impression
399997,399998,2,a08c1,2025-07-27 13:39:51,Click
399998,399999,109,4f0cf,2025-05-16 02:38:23,Impression


In [26]:
removed1 = ad_events.shape[0] -  ad_events_clean.shape[0]
print(f'{removed1} records were removed, {removed1*100/ad_events.shape[0]}% from the data')

4244 records were removed, 1.061% from the data


### step 3
Remove duplicate entries for the same user ID from the `users_interests` table.

In [27]:
user_interest_clean = user_interest.loc[~user_interest['user_id'].isin(duplicate_user_ids)]
user_interest_clean

,user_id,interest_id
0,a2474,12
1,a2474,6
2,14100000,7
3,14100000,12
4,14100000,4
...,...,...
19929,68b82,8
19930,39f39,6
19931,0b8e7,9
19932,0b8e7,1


In [29]:
removed2 = user_interest.shape[0] -  user_interest_clean.shape[0]
print(f'{removed2} records were removed, {round(removed2*100/user_interest.size,2)}% from the data')

213 records were removed, 0.53% from the data


### step 4
Remove records from the `ad_events` table with an event time earlier than the campaign start.

In [30]:
temp = ad_events_clean.merge(ads, on='ad_id')
temp = temp.merge(campaigns, on='campaign_id')
to_remove = temp.loc[temp['timestamp'] < temp['start_date']]['event_id']
ad_events_clean1 = ad_events_clean.loc[~ad_events_clean['event_id'].isin(to_remove)]
ad_events_clean1

,event_id,ad_id,user_id,timestamp,event_type
0,1,197,2359b,2025-07-26 00:19:56,Like
1,2,51,f9c67,2025-06-15 08:28:07,Share
2,3,46,5b868,2025-06-27 00:40:02,Impression
3,4,166,3d440,2025-06-05 19:20:45,Impression
4,5,52,68f1a,2025-07-22 08:30:29,Impression
...,...,...,...,...,...
399995,399996,132,3cb8c,2025-08-01 22:36:54,Impression
399996,399997,200,fe0e3,2025-05-31 14:53:18,Impression
399997,399998,2,a08c1,2025-07-27 13:39:51,Click
399998,399999,109,4f0cf,2025-05-16 02:38:23,Impression


In [31]:
removed3 = ad_events_clean.shape[0] -  ad_events_clean1.shape[0] 
print(f'{removed3} records were removed, {removed3*100/ad_events.shape[0]}% from the data')

2820 records were removed, 0.705% from the data


In [32]:
tables = [users_clean, campaigns, ads, interests, ad_events_clean1, user_interest_clean, ad_interest, sub_interests]
sheet_names = ['users', 'campaigns', 'ads', 'interests', 'ad_events', 'user_interest',
               'ad_interest', 'sub_interests']
path = 'Social_Media_Campaign_Analysis_Clean.xlsx'
er =pd.ExcelWriter(path)
for i in range(len(tables)):
    tables[i].to_excel(er, sheet_name=sheet_names[i], index = False)
er.close()

# Data Segmentation

Uplode Data

In [3]:
users = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis_Clean.xlsx', sheet_name='users')
ads = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis_Clean.xlsx', sheet_name='ads')
campaigns = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis_Clean.xlsx', sheet_name='campaigns')
ad_events = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis_Clean.xlsx', sheet_name='ad_events')
interests = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis_Clean.xlsx', sheet_name='interests')
user_interest = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis_Clean.xlsx', sheet_name='user_interest')
ad_interest = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis_Clean.xlsx', sheet_name='ad_interest')
sub_interests = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis_Clean.xlsx', sheet_name='sub_interests')


Splitting Age Ranges in the Ads Table

In [4]:
def extract_age(col_values, min_a, max_a):
    min_age_col,max_age_col = [], []

    for i in col_values:
        if i == 'All':
            min_age_col.append(min_a)
            max_age_col.append(max_a)
        else:
            min_age_col.append(int(i[:2]))
            max_age_col.append(int(i[-2:]))
    return min_age_col, max_age_col

min_age, max_age = min(users['user_age']), max(users['user_age'])
min_age_col, max_age_col = extract_age(ads['target_age_group'].values, min_age, max_age)
ads['min_age'] = min_age_col
ads['max_age'] = max_age_col
ads

,ad_id,campaign_id,ad_platform,ad_type,target_gender,target_age_group,min_age,max_age
0,1,28,Facebook,Video,Female,35-44,35,44
1,2,33,Facebook,Stories,All,25-34,25,34
2,3,20,Instagram,Carousel,All,25-34,25,34
3,4,28,Facebook,Stories,Female,25-34,25,34
4,5,24,Instagram,Image,Female,25-34,25,34
...,...,...,...,...,...,...,...,...
195,196,12,Facebook,Stories,Male,35-44,35,44
196,197,9,Facebook,Stories,All,All,16,65
197,198,34,Facebook,Video,All,35-44,35,44
198,199,15,Instagram,Video,Male,25-34,25,34


Categorizing Ads and Users into Age Groups

In [5]:
label_dict = {'All':'All', '18-24':'Young', '25-34':'Young Adults', '35-44':'Adults'}
ads['age_group_label'] = ads['target_age_group'].map(label_dict)
ads

,ad_id,campaign_id,ad_platform,ad_type,target_gender,target_age_group,min_age,max_age,age_group_label
0,1,28,Facebook,Video,Female,35-44,35,44,Adults
1,2,33,Facebook,Stories,All,25-34,25,34,Young Adults
2,3,20,Instagram,Carousel,All,25-34,25,34,Young Adults
3,4,28,Facebook,Stories,Female,25-34,25,34,Young Adults
4,5,24,Instagram,Image,Female,25-34,25,34,Young Adults
...,...,...,...,...,...,...,...,...,...
195,196,12,Facebook,Stories,Male,35-44,35,44,Adults
196,197,9,Facebook,Stories,All,All,16,65,All
197,198,34,Facebook,Video,All,35-44,35,44,Adults
198,199,15,Instagram,Video,Male,25-34,25,34,Young Adults


Categorizing Users into Age Groups

In [6]:
bins = [16, 25, 35, 45, 65]
labels = ['Young', 'Young Adults', 'Adults', 'Elderly']
users['age_label'] = pd.cut(users['user_age'],bins=bins,labels=labels,right=False,include_lowest=True)
users

,user_id,user_gender,user_age,country,location,age_label
0,a2474,Female,24,United Kingdom,New Mariomouth,Young
1,14100000,Male,21,Germany,Danielsfort,Young
2,34db0,Male,27,Australia,Vincentchester,Young Adults
3,20d08,Female,28,India,Lisaport,Young Adults
4,9e830,Male,28,United States,Brownmouth,Young Adults
...,...,...,...,...,...,...
9888,24364,Male,18,United States,Curtisside,Young
9889,68b82,Male,24,Mexico,Brownland,Young
9890,39f39,Male,29,United States,Watersburgh,Young Adults
9891,0b8e7,Male,31,United Kingdom,South Kenneth,Young Adults


Calculating Campaign Duration

In [7]:
campaigns['Duration (Days)'] = (campaigns['end_date'] - campaigns['start_date']).dt.days
campaigns

,campaign_id,name,start_date,end_date,total_budget,Duration (Days)
0,1,Campaign_1_Launch,2025-03-25,2025-07-23,24021.32,120
1,2,Campaign_2_Launch,2025-04-16,2025-07-07,79342.41,82
2,3,Campaign_3_Winter,2025-05-04,2025-06-29,14343.25,56
3,4,Campaign_4_Summer,2025-04-04,2025-08-08,45326.60,126
4,5,Campaign_5_Launch,2025-05-11,2025-08-28,68376.69,109
5,6,Campaign_6_Winter,2025-04-21,2025-09-13,78607.49,145
6,7,Campaign_7_Winter,2025-03-25,2025-08-05,43744.59,133
7,8,Campaign_8_Q3,2025-02-25,2025-04-07,39953.19,41
8,9,Campaign_9_Launch,2025-03-25,2025-07-13,40094.07,110
9,10,Campaign_10_Winter,2025-03-17,2025-07-21,19669.27,126


Extracting Date and Time Features

In [8]:
ad_events['Day_Of_Week'] = ad_events['timestamp'].dt.day_name()
ad_events['Is_weekend'] = ad_events['timestamp'].dt.dayofweek >= 5
ad_events['Hour'] = ad_events['timestamp'].dt.hour
ad_events

,event_id,ad_id,user_id,timestamp,event_type,Day_Of_Week,Is_weekend,Hour
0,1,197,2359b,2025-07-26 00:19:56,Like,Saturday,True,0
1,2,51,f9c67,2025-06-15 08:28:07,Share,Sunday,True,8
2,3,46,5b868,2025-06-27 00:40:02,Impression,Friday,False,0
3,4,166,3d440,2025-06-05 19:20:45,Impression,Thursday,False,19
4,5,52,68f1a,2025-07-22 08:30:29,Impression,Tuesday,False,8
...,...,...,...,...,...,...,...,...
392931,399996,132,3cb8c,2025-08-01 22:36:54,Impression,Friday,False,22
392932,399997,200,fe0e3,2025-05-31 14:53:18,Impression,Saturday,True,14
392933,399998,2,a08c1,2025-07-27 13:39:51,Click,Sunday,True,13
392934,399999,109,4f0cf,2025-05-16 02:38:23,Impression,Friday,False,2


In [9]:
from datetime import time

def get_day_part_from_time(t, day_part_dict):
    h = t
    for (start, end), label in day_part_dict.items():
        if start <= end:
            if start <= h <= end:
                return label
        else:
            if h >= start or h <= end:
                return label
    return None

In [10]:
day_part = {(6,11):'morning',(12,16):'noon', (17,22):'evening',(23,5):'night'}
ad_events['day_part'] = ad_events['Hour'].apply(
    lambda h: get_day_part_from_time(h, day_part)
)
ad_events

,event_id,ad_id,user_id,timestamp,event_type,Day_Of_Week,Is_weekend,Hour,day_part
0,1,197,2359b,2025-07-26 00:19:56,Like,Saturday,True,0,night
1,2,51,f9c67,2025-06-15 08:28:07,Share,Sunday,True,8,morning
2,3,46,5b868,2025-06-27 00:40:02,Impression,Friday,False,0,night
3,4,166,3d440,2025-06-05 19:20:45,Impression,Thursday,False,19,evening
4,5,52,68f1a,2025-07-22 08:30:29,Impression,Tuesday,False,8,morning
...,...,...,...,...,...,...,...,...,...
392931,399996,132,3cb8c,2025-08-01 22:36:54,Impression,Friday,False,22,evening
392932,399997,200,fe0e3,2025-05-31 14:53:18,Impression,Saturday,True,14,noon
392933,399998,2,a08c1,2025-07-27 13:39:51,Click,Sunday,True,13,noon
392934,399999,109,4f0cf,2025-05-16 02:38:23,Impression,Friday,False,2,night


# All Data

In [20]:
users = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis.xlsx', sheet_name='users')
users

,user_id,user_gender,user_age,country,location
0,a2474,Female,24,United Kingdom,New Mariomouth
1,14100000,Male,21,Germany,Danielsfort
2,34db0,Male,27,Australia,Vincentchester
3,20d08,Female,28,India,Lisaport
4,9e830,Male,28,United States,Brownmouth
...,...,...,...,...,...
9995,24364,Male,18,United States,Curtisside
9996,68b82,Male,24,Mexico,Brownland
9997,39f39,Male,29,United States,Watersburgh
9998,0b8e7,Male,31,United Kingdom,South Kenneth


In [21]:
ads = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis.xlsx', sheet_name='ads')
ads

,ad_id,campaign_id,ad_platform,ad_type,target_gender,target_age_group
0,1,28,Facebook,Video,Female,35-44
1,2,33,Facebook,Stories,All,25-34
2,3,20,Instagram,Carousel,All,25-34
3,4,28,Facebook,Stories,Female,25-34
4,5,24,Instagram,Image,Female,25-34
...,...,...,...,...,...,...
195,196,12,Facebook,Stories,Male,35-44
196,197,9,Facebook,Stories,All,All
197,198,34,Facebook,Video,All,35-44
198,199,15,Instagram,Video,Male,25-34


In [3]:
campaigns = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis.xlsx', sheet_name='campaigns')
campaigns

,campaign_id,name,start_date,end_date,total_budget
0,1,Campaign_1_Launch,2025-03-25,2025-07-23,24021.32
1,2,Campaign_2_Launch,2025-04-16,2025-07-07,79342.41
2,3,Campaign_3_Winter,2025-05-04,2025-06-29,14343.25
3,4,Campaign_4_Summer,2025-04-04,2025-08-08,45326.60
4,5,Campaign_5_Launch,2025-05-11,2025-08-28,68376.69
5,6,Campaign_6_Winter,2025-04-21,2025-09-13,78607.49
6,7,Campaign_7_Winter,2025-03-25,2025-08-05,43744.59
7,8,Campaign_8_Q3,2025-02-25,2025-04-07,39953.19
8,9,Campaign_9_Launch,2025-03-25,2025-07-13,40094.07
9,10,Campaign_10_Winter,2025-03-17,2025-07-21,19669.27


In [20]:
ad_events = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis.xlsx', sheet_name='ad_events')
ad_events

,event_id,ad_id,user_id,timestamp,event_type
0,1,197,2359b,2025-07-26 00:19:56,Like
1,2,51,f9c67,2025-06-15 08:28:07,Share
2,3,46,5b868,2025-06-27 00:40:02,Impression
3,4,166,3d440,2025-06-05 19:20:45,Impression
4,5,52,68f1a,2025-07-22 08:30:29,Impression
...,...,...,...,...,...
399995,399996,132,3cb8c,2025-08-01 22:36:54,Impression
399996,399997,200,fe0e3,2025-05-31 14:53:18,Impression
399997,399998,2,a08c1,2025-07-27 13:39:51,Click
399998,399999,109,4f0cf,2025-05-16 02:38:23,Impression


In [21]:
interests = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis.xlsx', sheet_name='interests')
interests

,interest_id,interest
0,1,art
1,2,technology
2,3,food
3,4,fitness
4,5,gaming
5,6,finance
6,7,lifestyle
7,8,travel
8,9,sports
9,10,health


In [22]:
user_interest = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis.xlsx', sheet_name='user_interest')
user_interest

,user_id,interest_id
0,a2474,4
1,a2474,10
2,14100000,3
3,14100000,4
4,14100000,7
...,...,...
19929,68b82,6
19930,39f39,10
19931,0b8e7,1
19932,0b8e7,12


In [23]:
ad_interest = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis.xlsx', sheet_name='ad_interest')
ad_interest

,ad_id,interest_id
0,1,1
1,1,2
2,2,8
3,2,13
4,3,2
...,...,...
295,198,4
296,198,7
297,199,1
298,199,11


In [24]:
sub_interests = pd.read_excel('/data/notebook_files/Social_Media_Campaign_Analysis.xlsx', sheet_name='sub_interests')
sub_interests

,interest_id,interest_id_related
0,1,12
1,1,13
2,2,6
3,2,5
4,2,11
5,2,13
6,3,7
7,3,10
8,3,4
9,4,3
